# 👯 K-Nearest Neighbors (KNN) - In Depth

K-Nearest Neighbors is often the very first classification algorithm taught in Machine Learning because it perfectly mimics human intuition.

Imagine you move to a new city and want to know if a specific neighborhood is safe. What do you do? You look at the houses immediately surrounding it. If the 5 closest neighborhoods are safe, you assume yours is too. This is exactly what KNN does!

## 🧠 1. Deep Dive into the Theory

KNN is a **Non-Parametric, Lazy Learning** algorithm.

1. **Non-Parametric**: It makes zero assumptions about the underlying data distribution. Linear Regression assumes a straight line. KNN assumes nothing. This makes it incredibly flexible for messy real-world data.
2. **Lazy Learning**: Most algorithms spend minutes or hours *training* (finding weights, slopes, and intercepts). KNN does absolutely nothing during the training phase. It just stores the data in memory. The heavy lifting only happens when you ask it to make a prediction.

## 🧮 2. The Math: Calculating Distance by Hand

To find the "nearest" neighbors, we need a mathematical definition of distance.

### Euclidean Distance (The Crow's Flight)
The straight-line distance between two points $(x_1, y_1)$ and $(x_2, y_2)$.
$$ d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2} $$

**Example Calculation:**
Suppose we are classifying fruits based on Weight (x) and Sweetness (y).
- Apple: $(150g, 8)$
- Orange: $(130g, 6)$
- **Unknown Fruit**: $(140g, 7)$

Distance to Apple: $\sqrt{(150 - 140)^2 + (8 - 7)^2} = \sqrt{100 + 1} = \sqrt{101} \approx 10.05$
Distance to Orange: $\sqrt{(130 - 140)^2 + (6 - 7)^2} = \sqrt{100 + 1} = \sqrt{101} \approx 10.05$

*(Since they are equidistant, we'd need more neighbors to break the tie!)*

## 💻 3. Implementation and The Curse of Dimensionality

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Generate a complex dataset
X, y = make_classification(n_samples=500, n_features=2, n_informative=2, n_redundant=0, 
                           n_clusters_per_class=1, flip_y=0.1, class_sep=1.5, random_state=42)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X[:,0], y=X[:,1], hue=y, palette='coolwarm', edgecolor='k')
plt.title('Complex Classification Data (With Noise)')
plt.show()

### Feature Scaling is NOT Optional!
If Feature A ranges from 0 to 1, and Feature B ranges from 0 to 1,000,000, the Euclidean distance will be entirely dominated by Feature B. The algorithm will act as if Feature A doesn't even exist!

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 🔍 4. Finding the Perfect 'K' using Cross-Validation

If $K=1$, the model overfits (it memorizes the noise). If $K=N$ (total samples), the model underfits (it just predicts the majority class). We use **Cross-Validation** to find the sweet spot.

In [ ]:
k_values = list(range(1, 30, 2)) # Use odd numbers to prevent ties
cv_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    # 5-fold cross validation on training data
    scores = cross_val_score(knn, X_train_scaled, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

optimal_k = k_values[np.argmax(cv_scores)]
print(f"The optimal number of neighbors is: {optimal_k}")

plt.figure(figsize=(10, 6))
plt.plot(k_values, cv_scores, marker='o', linestyle='dashed', color='blue', markerfacecolor='red')
plt.title('Cross-Validation Accuracy vs. K Value')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Cross-Validation Accuracy')
plt.axvline(x=optimal_k, color='green', linestyle='--')
plt.show()

## 🚀 5. Final Evaluation and Decision Boundaries

In [ ]:
# Train the final model with the optimal K
final_knn = KNeighborsClassifier(n_neighbors=optimal_k)
final_knn.fit(X_train_scaled, y_train)

y_pred = final_knn.predict(X_test_scaled)
print("Final Model Performance:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Visualize the decision boundary
x_min, x_max = X_train_scaled[:, 0].min() - 1, X_train_scaled[:, 0].max() + 1
y_min, y_max = X_train_scaled[:, 1].min() - 1, X_train_scaled[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))

Z = final_knn.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=y_train, s=20, edgecolor='k', cmap='coolwarm')
plt.title(f'KNN Decision Boundary (K={optimal_k})')
plt.show()

## 📊 6. Summary: Pros and Cons

| Pros | Cons |
|------|------|
| Extremely simple and intuitive | Very slow at prediction time on large datasets |
| No assumptions about data distribution | Highly sensitive to irrelevant features |
| Naturally handles multi-class problems | **The Curse of Dimensionality**: Fails when you have hundreds of features (distances lose meaning in high dimensions) |